# FT-Transformer for 24-Hour Deterioration Early Warning in the ICU (MIMIC-IV v3.1)
**Author:** Nabil Walid Rafi — Department of ICT, FST, Bangladesh University of Professionals

**Task.** From data recorded in the first 24 h after ICU admission, predict the patient's official
NEWS2 clinical-risk category (Royal College of Physicians, 2017) over the next 24 h (24–48 h).
Death within 24–48 h counts as High.

**Pipeline.** slim MIMIC-IV extract → two time windows → ≈60 features (vital-sign summaries, GCS, labs, demographics,
care unit, ventilation, vasopressors) → patient-level split → hyperparameters tuned with Optuna on the validation split
(`tune_hyperparameters.ipynb`) → FT-Transformer vs NEWS2 persistence, logistic regression, random forest, XGBoost,
LightGBM → patient-clustered bootstrap CIs → High risk or death at 24–48 h (AUROC, AUPRC, accuracy, sensitivity,
specificity at validation-chosen thresholds) → permutation importance (FT-Transformer) + SHAP (LightGBM) →
sensitivity analyses (no class weighting; three categories; 24–30 h outcome).

Inputs: the private `mimic-slim` dataset made by `make_mimic_slim.ipynb` (or the raw CSVs) and, optionally, the output
of `tune_hyperparameters.ipynb` (`best_params.json`; published default settings are used if it is not attached).
Outputs: 300-dpi PNG + PDF figures and CSV tables in the working directory.

Citation: Johnson A, et al. MIMIC-IV (version 3.1). PhysioNet (2024). https://doi.org/10.13026/kpb9-mt58

## 1. Setup

In [ ]:
import os, glob, gc, copy, math, random, time, warnings
import builtins
_print = builtins.print
def print(*a, **k): k.setdefault('flush', True); _print(*a, **k)   # every line reaches the Kaggle log immediately
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, matplotlib.patches as mpatches, seaborn as sns
from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (average_precision_score, precision_recall_curve, accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score,
                             roc_auc_score, roc_curve, auc, confusion_matrix, classification_report)
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgbm
import shap, torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FTT_EPOCHS   = int(os.environ.get('FTT_EPOCHS', 200))     # maximum; early stopping usually ends much sooner
FTT_PATIENCE = int(os.environ.get('FTT_PATIENCE', 16))    # epochs without validation improvement
N_SEEDS      = int(os.environ.get('N_SEEDS', 5))          # FT-Transformer seed ensemble
FTT_EMB      = os.environ.get('FTT_EMB', 'plr')           # 'plr' (periodic, Gorishniy 2022) or 'linear'
N_SEEDS_SENS = int(os.environ.get('N_SEEDS_SENS', 3))      # FT-Transformer seeds in each sensitivity analysis
RUN_SENS     = os.environ.get('RUN_SENS', '1') == '1'      # run sensitivity analyses C–E
SHORT_H      = 30                                          # sensitivity analysis E: outcome window 24–30 h
N_BOOT     = int(os.environ.get('N_BOOT', 1000))
N_PERM     = int(os.environ.get('N_PERM', 5))
OBS_H, PRED_H = 24, 48

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9, 'axes.edgecolor': '#52514e', 'text.color': '#0b0b0b',
    'xtick.color': '#52514e', 'ytick.color': '#52514e', 'legend.frameon': False})
CLASSES     = ['Low', 'Low–medium', 'Medium', 'High']        # official NEWS2 clinical-risk categories
CLASS_COLORS = ['#86b6ef', '#3987e5', '#1c5cab', '#0d366b']    # ordinal blue ramp (validated)
C_BLUE, C_ORANGE, C_AQUA, C_GREY = '#2a78d6', '#eb6834', '#1baf7a', '#9a9893'

FIG_DIR = os.environ.get('FIG_DIR', '.'); os.makedirs(FIG_DIR, exist_ok=True)
def save_fig(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}.png', dpi=300, bbox_inches='tight', facecolor='white')
    fig.savefig(f'{FIG_DIR}/{name}.pdf', bbox_inches='tight', facecolor='white'); print(f'✅ {name}.png/.pdf')
def save_table(df, name):
    df.to_csv(f'{FIG_DIR}/{name}.csv', index=False); print(f'✅ {name}.csv')

# ── Official NEWS2 scoring (RCP 2017; SpO2 scale 1) ───────────────────────────
VITALS = ['RR', 'SpO2', 'HR', 'SBP', 'Temp']
def news2_parts(rr, spo2, o2, sbp, hr, cons, temp):
    rr, spo2, sbp, hr, temp = (np.asarray(x, float) for x in (rr, spo2, sbp, hr, temp))
    cons, o2 = np.asarray(cons, float), np.asarray(o2, float)
    s = {
      'RR':   np.select([rr <= 8, rr <= 11, rr <= 20, rr <= 24, rr > 24], [3, 1, 0, 2, 3], 0),
      'SpO2': np.select([spo2 <= 91, spo2 <= 93, spo2 <= 95], [3, 2, 1], 0),
      'O2':   np.where(o2 == 1, 2, 0),
      'SBP':  np.select([sbp <= 90, sbp <= 100, sbp <= 110, sbp <= 219, sbp >= 220], [3, 2, 1, 0, 3], 0),
      'HR':   np.select([hr <= 40, hr <= 50, hr <= 90, hr <= 110, hr <= 130, hr > 130], [3, 1, 0, 1, 2, 3], 0),
      'Cons': np.where(cons > 0, 3, 0),
      'Temp': np.select([temp <= 35.0, temp <= 36.0, temp <= 38.0, temp <= 39.0, temp > 39.0], [3, 1, 0, 1, 2], 0)}
    total = sum(s.values()); red = np.max(np.stack(list(s.values())), axis=0) == 3
    return total.astype(int), red
def news2_category(total, red):
    # High ≥7; Medium 5–6; Low–medium = a single parameter scoring 3; Low otherwise (0–4)
    return np.select([total >= 7, total >= 5, red], [3, 2, 1], 0)
print(f'device={device} | FT-T max epochs={FTT_EPOCHS}, patience={FTT_PATIENCE}, seeds={N_SEEDS}, embedding={FTT_EMB} | bootstrap={N_BOOT}')


## 2. Locate inputs (slim parquet preferred; raw CSV fallback)

In [ ]:
DATA_ROOT = os.environ.get('DATA_ROOT', '/kaggle/input')
def find(*patterns):
    for p in patterns:
        hits = sorted(glob.glob(os.path.join(DATA_ROOT, '**', p), recursive=True)) + sorted(glob.glob(p))
        if hits: return hits[0]
    return None
P_ICU = find('icustays.parquet', 'icustays.csv*')
P_ADM = find('admissions.parquet', 'admissions.csv*')
P_CE  = find('chartevents_vitals_48h.parquet', 'chartevents_vitals_filtered.csv*')
P_PAT = find('patients.parquet')            # optional extras (skipped with a warning if absent)
P_LAB = find('labs_48h.parquet')
P_PRO = find('procedures_vent.parquet')
P_INP = find('inputs_vaso.parquet')
for k, v in [('icustays', P_ICU), ('admissions', P_ADM), ('chartevents', P_CE), ('patients', P_PAT),
             ('labs', P_LAB), ('ventilation', P_PRO), ('vasopressors', P_INP)]: print(f'{k:12s}: {v}')
for k, v in [('patients', P_PAT), ('labs', P_LAB), ('ventilation', P_PRO), ('vasopressors', P_INP)]:
    if not v: print(f'⚠ {k} not found — those features will be skipped (re-run make_mimic_slim to add them)')
assert P_ICU and P_ADM and P_CE, 'Attach the private mimic-slim dataset (see make_mimic_slim.ipynb).'


## 3. Load, harmonise and tag time windows

In [ ]:
rd = lambda p, **kw: pd.read_parquet(p) if p.endswith('.parquet') else pd.read_csv(p, **kw)
icu = rd(P_ICU)
adm = rd(P_ADM)
adm = adm[[c for c in ['subject_id', 'hadm_id', 'admission_type', 'deathtime', 'hospital_expire_flag'] if c in adm.columns]]
stays = icu.merge(adm, on=['subject_id', 'hadm_id'], how='left')
for c in ['intime', 'outtime', 'deathtime']: stays[c] = pd.to_datetime(stays[c], errors='coerce')

ce = rd(P_CE)
HAS_DEVICE_TEXT = 'value' in ce.columns
ce['charttime'] = pd.to_datetime(ce['charttime'], errors='coerce')
ce = ce.merge(stays[['stay_id', 'intime']], on='stay_id', how='inner')
ce['h'] = (ce['charttime'] - ce['intime']).dt.total_seconds() / 3600
ce = ce[(ce['h'] >= 0) & (ce['h'] < PRED_H)].sort_values(['stay_id', 'charttime'])
ce['window'] = np.where(ce['h'] < OBS_H, 'obs', 'pred')

ITEM = {220210: 'RR', 220277: 'SpO2', 220045: 'HR', 220179: 'SBP', 220050: 'SBP', 223762: 'Temp', 223761: 'TempF',
        220739: 'GCS_eye', 223900: 'GCS_verbal', 223901: 'GCS_motor', 220181: 'MAP', 220052: 'MAP',
        226732: 'O2_device', 223834: 'O2_flow'}   # SBP: non-invasive or arterial; MAP: non-invasive or arterial
ce['var'] = ce['itemid'].map(ITEM); ce = ce.dropna(subset=['var'])
f = ce['var'] == 'TempF'
ce.loc[f, 'valuenum'] = (ce.loc[f, 'valuenum'] - 32) * 5 / 9; ce.loc[f, 'var'] = 'Temp'

# Consciousness from GCS eye opening → ACVPU-style code: 4 spontaneous→0 (alert), 3→2 (voice), 2→3 (pain), 1→4 (unresponsive)
g = ce['var'] == 'GCS_eye'
ce.loc[g, 'valuenum'] = ce.loc[g, 'valuenum'].round().map({4: 0, 3: 2, 2: 3, 1: 4})
ce.loc[g, 'var'] = 'Cons'

# Supplemental oxygen: a charted device other than 'None', or O2 flow > 0 L/min
o = ce['var'] == 'O2_device'
if HAS_DEVICE_TEXT:
    txt = ce.loc[o, 'value'].astype('string').str.strip().str.lower()
    ce.loc[o, 'valuenum'] = np.where(txt.isin(['none', 'room air', 'ra']) | txt.isna(), 0.0, 1.0)
else:
    print('⚠ No device text in input — any charted O2 device is treated as oxygen (re-run make_mimic_slim).')
    ce.loc[o, 'valuenum'] = 1.0
fl = ce['var'] == 'O2_flow'
ce.loc[fl, 'valuenum'] = (ce.loc[fl, 'valuenum'] > 0).astype(float)
ce.loc[o | fl, 'var'] = 'O2'

VALID = {'RR': (4, 60), 'SpO2': (50, 100), 'HR': (20, 250), 'SBP': (40, 260), 'Temp': (30, 43), 'MAP': (20, 200),
         'GCS_verbal': (1, 5), 'GCS_motor': (1, 6)}
bad = np.zeros(len(ce), bool)
for v, (lo, hi) in VALID.items():
    m = (ce['var'] == v).values
    bad |= m & ~ce['valuenum'].between(lo, hi).values
ce = ce[~bad].dropna(subset=['valuenum'])
print(f'Artefacts removed: {bad.sum():,} | rows kept: {len(ce):,} | stays: {ce.stay_id.nunique():,}')


## 4. Window summaries

In [ ]:
agg = (ce.groupby(['stay_id', 'window', 'var'])['valuenum']
         .agg(['median', 'min', 'max', 'std', 'first', 'last', 'count']).reset_index())
def wide(win, stats):
    a = agg[agg['window'] == win].pivot(index='stay_id', columns='var', values=stats)
    a.columns = [f'{v}_{s}' for s, v in a.columns]; return a
obs  = wide('obs',  ['median', 'min', 'max', 'std', 'first', 'last', 'count'])
pred = wide('pred', ['median', 'max', 'count'])
for d, cols in [(obs, [f'{v}_{s}' for v in VITALS + ['Cons', 'O2', 'MAP', 'GCS_verbal', 'GCS_motor'] for s in ['median', 'min', 'max', 'std', 'first', 'last', 'count']]),
                (pred, [f'{v}_{s}' for v in VITALS + ['Cons', 'O2'] for s in ['median', 'max', 'count']])]:
    for c in cols:
        if c not in d.columns: d[c] = np.nan
c6 = ce[(ce['h'] >= OBS_H) & (ce['h'] < SHORT_H)]              # sensitivity analysis E: 24–30 h
pred6 = c6.groupby(['stay_id', 'var'])['valuenum'].agg(['median', 'max', 'count']).unstack('var')
pred6.columns = [f'{v}_{s}' for s, v in pred6.columns]
for c in [f'{v}_{s}' for v in VITALS + ['Cons', 'O2'] for s in ['median', 'max', 'count']]:
    if c not in pred6.columns: pred6[c] = np.nan
print(f'obs-window stays: {len(obs):,} | pred-window stays: {len(pred):,} | 24–30 h stays: {len(pred6):,}')
del ce, agg, c6; gc.collect()


## 5. Outcome (official NEWS2 category at 24–48 h), cohort and patient-level split
Prediction-window NEWS2 uses per-stay medians of RR, SpO₂, SBP, HR and temperature, the median consciousness
code, and supplemental oxygen if any was charted. A parameter not recorded at 24–48 h is carried forward
from 0–24 h. Death between 24 and 48 h → High.

In [ ]:
flow = {'ICU stays (MIMIC-IV v3.1)': len(stays), 'Unique patients': stays.subject_id.nunique()}
df = stays[['subject_id', 'stay_id', 'intime', 'deathtime']].merge(obs, left_on='stay_id', right_index=True)
flow['With 0–24 h vital signs'] = len(df)
df = df[df[[f'{v}_median' for v in VITALS]].notnull().sum(axis=1) >= 3]
flow['≥3 of 5 core vitals at 0–24 h'] = len(df)
hrs_death = (df['deathtime'] - df['intime']).dt.total_seconds() / 3600
df = df[~(hrs_death < OBS_H)]
flow['Alive at 24 h'] = len(df)
died = ((df['deathtime'] - df['intime']).dt.total_seconds() / 3600).between(OBS_H, PRED_H, inclusive='left').values

P = pred.reindex(df['stay_id'])
measured = P[[f'{v}_median' for v in VITALS]].notnull().sum(axis=1).values
keep = (measured >= 3) | died
df, P, died = df[keep].reset_index(drop=True), P[keep].reset_index(drop=True), died[keep]
flow['Outcome available at 24–48 h (final cohort)'] = len(df)

def locf(pcol, ocol): return P[pcol].fillna(pd.Series(df[ocol].values)).values
tot, red = news2_parts(locf('RR_median', 'RR_median'), locf('SpO2_median', 'SpO2_median'),
                       np.where(P['O2_max'].notnull(), P['O2_max'], df['O2_max'].values),
                       locf('SBP_median', 'SBP_median'), locf('HR_median', 'HR_median'),
                       np.round(locf('Cons_median', 'Cons_median')), locf('Temp_median', 'Temp_median'))
df['NEWS2_24_48'] = tot
df['y'] = news2_category(tot, red)
df.loc[died, 'y'] = 3
df['died_24_48'] = died.astype(int)

# Sensitivity analysis E: NEWS2 category over 24–30 h, within the same cohort (≥3 core vitals at 24–30 h or death in 24–30 h)
P6 = pred6.reindex(df['stay_id']).reset_index(drop=True)
died6 = ((df['deathtime'] - df['intime']).dt.total_seconds() / 3600).between(OBS_H, SHORT_H, inclusive='left').values
ok6 = (P6[[f'{v}_median' for v in VITALS]].notnull().sum(axis=1).values >= 3) | died6
def locf6(c): return P6[c].fillna(pd.Series(df[c].values)).values
t6, r6 = news2_parts(locf6('RR_median'), locf6('SpO2_median'), np.where(P6['O2_max'].notnull(), P6['O2_max'], df['O2_max'].values),
                     locf6('SBP_median'), locf6('HR_median'), np.round(locf6('Cons_median')), locf6('Temp_median'))
df['y_6h'] = np.where(ok6, news2_category(t6, r6), np.nan); df.loc[died6, 'y_6h'] = 3

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
tr_i, te_i = next(sgkf.split(df, df['y'], groups=df['subject_id']))
df['split'] = 'train'; df.loc[te_i, 'split'] = 'test'
assert set(df.loc[df.split == 'train', 'subject_id']).isdisjoint(df.loc[df.split == 'test', 'subject_id'])
flow['Train stays / patients'] = f"{(df.split=='train').sum():,} / {df.loc[df.split=='train','subject_id'].nunique():,}"
flow['Test stays / patients']  = f"{(df.split=='test').sum():,} / {df.loc[df.split=='test','subject_id'].nunique():,}"
for k, v in flow.items(): print(f'  {k:45s}: {v:,}' if isinstance(v, (int, np.integer)) else f'  {k:45s}: {v}')
for k in range(4):
    n = (df.y == k).sum(); print(f'  {CLASSES[k]:11s}: {n:7,} ({n/len(df)*100:.1f}%)')
print(f'  deaths 24–48 h: {df.died_24_48.sum():,} | stays with a 24–30 h outcome (analysis E): {df.y_6h.notnull().sum():,}')


## 5b. Context, laboratory values and treatments (0–24 h)
Age and sex (patients), emergency admission (admissions), first ICU care unit (icustays); laboratory values from
6 h before to 24 h after ICU admission (worst value: max or min); invasive or non-invasive ventilation and
vasopressor infusion active at any time in 0–24 h. Missing sources are skipped with a warning.

In [ ]:
LAB_DEF = {50813: ('lactate', 'max'), 50912: ('creatinine', 'max'), 51006: ('bun', 'max'), 51301: ('wbc', 'max'),
           51300: ('wbc', 'max'), 51265: ('platelets', 'min'), 50885: ('bilirubin', 'max'), 51222: ('hemoglobin', 'min'),
           50882: ('bicarbonate', 'min'), 50971: ('potassium', 'max'), 50931: ('glucose', 'max'), 50983: ('sodium', 'min')}
LAB_F, CTX_F, TRT_F = [], [], []
if P_PAT:
    pat = rd(P_PAT)[['subject_id', 'gender', 'anchor_age', 'anchor_year']]
    m = df[['subject_id', 'intime']].merge(pat, on='subject_id', how='left')
    df['age'] = (m['anchor_age'] + (m['intime'].dt.year - m['anchor_year'])).values.astype(float)
    df['male'] = (m['gender'] == 'M').astype(float).values
    CTX_F += ['age', 'male']
st = stays.set_index('stay_id')
if 'admission_type' in st.columns:
    at = df['stay_id'].map(st['admission_type']).astype('string').str.upper()
    df['emergency_admission'] = at.str.contains('EMER|URGENT', na=False).astype(float).values
    CTX_F.append('emergency_admission')
if 'first_careunit' in st.columns:
    def unit_group(u):
        u = str(u)
        if 'Neuro' in u: return 'neuro'
        if 'Cardiac' in u or 'CCU' in u or 'Coronary' in u: return 'cardiac'
        if 'Medical' in u or 'MICU' in u: return 'medical'
        if 'Surgical' in u or 'SICU' in u or 'Trauma' in u: return 'surgical'
        return 'other'
    ug = df['stay_id'].map(st['first_careunit']).map(unit_group)
    for g in ['cardiac', 'surgical', 'neuro', 'other']:          # medical ICU = reference
        df[f'unit_{g}'] = (ug == g).astype(float).values; CTX_F.append(f'unit_{g}')
if P_LAB:
    lab = rd(P_LAB)
    lab = lab[(lab['hours_from_intime'] >= -6) & (lab['hours_from_intime'] < OBS_H) & (lab['valuenum'] > 0)]
    lab['name'] = lab['itemid'].map({k: v[0] for k, v in LAB_DEF.items()})
    g = lab.dropna(subset=['name']).groupby(['stay_id', 'name'])['valuenum']
    lmax, lmin = g.max(), g.min()
    for name, how in {v[0]: v[1] for v in LAB_DEF.values()}.items():
        src = (lmax if how == 'max' else lmin)
        src = src[src.index.get_level_values('name') == name].droplevel('name')
        df[f'lab_{name}'] = df['stay_id'].map(src).values; LAB_F.append(f'lab_{name}')
    df['lactate_measured'] = df['lab_lactate'].notnull().astype(float); LAB_F.append('lactate_measured')
    print('lab availability (% of stays):', {c: round(df[c].notnull().mean() * 100, 1) for c in LAB_F if c != 'lactate_measured'})
def active_0_24(path, items=None):
    e = rd(path); e = e[(e['start_h'] < OBS_H) & (e['end_h'] >= 0)]
    if items: e = e[e['itemid'].isin(items)]
    return df['stay_id'].isin(e['stay_id']).astype(float).values
if P_PRO:
    df['vent_invasive'] = active_0_24(P_PRO, [225792]); df['vent_noninvasive'] = active_0_24(P_PRO, [225794]); TRT_F += ['vent_invasive', 'vent_noninvasive']
if P_INP:
    df['vasopressor'] = active_0_24(P_INP); TRT_F.append('vasopressor')
print(f'context: {CTX_F}\nlabs: {LAB_F}\ntreatments: {TRT_F}')


## 6. Feature set (0–24 h)
Vital-sign summaries (RR, SpO₂, HR, SBP, temperature: median, min, max, SD, last, change), MAP (median, min, last),
GCS verbal and motor (min), consciousness (worst, last), supplemental O₂, NEWS2 at 0–24 h, laboratory worst values,
age, sex, emergency admission, care unit, ventilation and vasopressors. Training-patient medians impute missing
continuous values (SD/change → 0; flags → 0).

In [ ]:
for v in VITALS:
    df[f'{v}_delta'] = df[f'{v}_last'] - df[f'{v}_first']
df['Cons_worst'] = df['Cons_max']; df['O2_any'] = df['O2_max']
VITAL_F = [f'{v}_{s}' for v in VITALS for s in ['median', 'min', 'max', 'std', 'last', 'delta']]
EXTRA_VITAL_F = [c for c in ['MAP_median', 'MAP_min', 'MAP_last', 'GCS_verbal_min', 'GCS_motor_min'] if df[c].notnull().mean() > 0.05]
NEURO_O2_F = ['Cons_worst', 'Cons_last', 'O2_any', 'NEWS2_0_24']
FEATURES = VITAL_F + EXTRA_VITAL_F + NEURO_O2_F + LAB_F + CTX_F + TRT_F

tr_mask = (df.split == 'train').values
for c in [f'{v}_{s}' for v in VITALS for s in ['median', 'min', 'max', 'last']] + EXTRA_VITAL_F + [c for c in LAB_F if c.startswith('lab_')] + [c for c in ['age'] if c in df]:
    df[c] = df[c].fillna(df.loc[tr_mask, c].median())
for c in [f'{v}_{s}' for v in VITALS for s in ['std', 'delta']]: df[c] = df[c].fillna(0.0)
for c in ['Cons_worst', 'Cons_last', 'Cons_median', 'O2_any'] + [c for c in CTX_F + TRT_F if c != 'age']:
    df[c] = df[c].fillna(0)
t0, r0 = news2_parts(df['RR_median'], df['SpO2_median'], df['O2_any'], df['SBP_median'], df['HR_median'],
                     np.round(df['Cons_median']), df['Temp_median'])
df['NEWS2_0_24'] = t0; df['cat_0_24'] = news2_category(t0, r0)

# ── display names, permutation groups and colour domains ──
LABN = {'lactate': 'Lactate', 'creatinine': 'Creatinine', 'bun': 'Urea nitrogen', 'wbc': 'WBC', 'platelets': 'Platelets',
        'bilirubin': 'Bilirubin', 'hemoglobin': 'Haemoglobin', 'bicarbonate': 'Bicarbonate', 'potassium': 'Potassium',
        'glucose': 'Glucose', 'sodium': 'Sodium'}
VN = {'RR': 'RR', 'SpO2': 'SpO₂', 'HR': 'HR', 'SBP': 'SBP', 'Temp': 'Temp', 'MAP': 'MAP'}
INFO = {}
for f in FEATURES:
    if f.startswith('lab_'):
        n = f[4:]; INFO[f] = (f"{LABN[n]} ({'max' if LAB_DEF and dict((v[0], v[1]) for v in LAB_DEF.values())[n] == 'max' else 'min'})", 'Laboratory', 'Laboratory')
    elif f == 'lactate_measured': INFO[f] = ('Lactate measured', 'Laboratory', 'Laboratory')
    elif f.split('_')[0] in VN:
        v, st_ = f.split('_', 1); INFO[f] = (f"{VN[v]} {st_.replace('delta', 'change')}", 'Vital signs', v)
    elif f.startswith('GCS_'): INFO[f] = (f"GCS {f.split('_')[1]} (min)", 'Neurological & O₂', 'GCS & consciousness')
    elif f in ('Cons_worst', 'Cons_last'): INFO[f] = (f"Consciousness ({f.split('_')[1]})", 'Neurological & O₂', 'GCS & consciousness')
    elif f == 'O2_any': INFO[f] = ('Supplemental O₂', 'Neurological & O₂', 'Oxygen & ventilation')
    elif f == 'NEWS2_0_24': INFO[f] = ('NEWS2 (0–24 h)', 'NEWS2', 'NEWS2 (0–24 h)')
    elif f in TRT_F: INFO[f] = ({'vent_invasive': 'Invasive ventilation', 'vent_noninvasive': 'Non-invasive ventilation',
                                'vasopressor': 'Vasopressor'}[f], 'Treatment',
                               'Oxygen & ventilation' if f.startswith('vent') else 'Vasopressor')
    else: INFO[f] = ({'age': 'Age', 'male': 'Male sex', 'emergency_admission': 'Emergency admission'}.get(f, f.replace('unit_', 'Unit: ').capitalize()),
                     'Context', 'Demographics & unit')
def nice(f): return INFO[f][0]
GROUP = {f: INFO[f][2] for f in FEATURES}
DOMAIN = {f: INFO[f][1] for f in FEATURES}

tr, te = df[df.split == 'train'].reset_index(drop=True), df[df.split == 'test'].reset_index(drop=True)
X_tr, y_tr = tr[FEATURES].values.astype(float), tr['y'].values
X_te, y_te = te[FEATURES].values.astype(float), te['y'].values
scaler = StandardScaler().fit(X_tr)
X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)
# FT-Transformer input: quantile-normal transform of continuous features (Gorishniy et al., 2021), standardised flags;
# both fitted on training data only. Skewed labs (lactate, creatinine, bilirubin …) become well-scaled for the network.
CONT = np.array([len(np.unique(X_tr[:, j])) > 10 for j in range(X_tr.shape[1])])
qt = QuantileTransformer(output_distribution='normal', n_quantiles=min(1000, len(X_tr)), subsample=10**9,
                         random_state=SEED).fit(X_tr[:, CONT])
def to_q(X):
    Z = scaler.transform(X); Z[:, CONT] = qt.transform(X[:, CONT]); return Z.astype(np.float32)
X_tr_q, X_te_q = to_q(X_tr), to_q(X_te)
cw = compute_class_weight('balanced', classes=np.arange(4), y=y_tr)
# patient-grouped 10% validation split of the training set (early stopping and FT-Transformer checkpointing)
fit_i, val_i = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED).split(X_tr_s, y_tr, tr['subject_id']))
print(f'{len(FEATURES)} features | train {X_tr_s.shape} | test {X_te_s.shape} | class weights {np.round(cw, 3)}')
print('feature domains:', pd.Series(DOMAIN).value_counts().to_dict())
print(f'0–24 h category equals 24–48 h outcome for {(df.cat_0_24 == df.y).mean()*100:.1f}% of stays')


## 7. Cohort description

In [ ]:
save_table(pd.DataFrame({'Step': list(flow), 'n': [f'{v:,}' if isinstance(v, (int, np.integer)) else v for v in flow.values()]}),
           'table_1_cohort_flow')
rows = []
for k in range(4):
    r = {'NEWS2 category (24–48 h)': CLASSES[k]}
    for nm, part in [('All', df), ('Train', tr), ('Test', te)]:
        n = (part.y == k).sum(); r[f'{nm} n'] = n; r[f'{nm} %'] = round(n / len(part) * 100, 1)
    rows.append(r)
save_table(pd.DataFrame(rows), 'table_2_outcome_distribution')
def miqr(s): return f'{s.median():.1f} [{s.quantile(.25):.1f}–{s.quantile(.75):.1f}]'
ROWS = [('Age (years)', 'age', 'c'), ('Male (%)', 'male', 'p'), ('Emergency admission (%)', 'emergency_admission', 'p'),
        ('RR (/min)', 'RR_median', 'c'), ('SpO₂ (%)', 'SpO2_median', 'c'), ('HR (bpm)', 'HR_median', 'c'),
        ('SBP (mmHg)', 'SBP_median', 'c'), ('Temperature (°C)', 'Temp_median', 'c'), ('NEWS2 total', 'NEWS2_0_24', 'c'),
        ('Supplemental O₂ (%)', 'O2_any', 'p'), ('Lactate, max (mmol/L)', 'lab_lactate', 'c'), ('Creatinine, max (mg/dL)', 'lab_creatinine', 'c'),
        ('Invasive ventilation (%)', 'vent_invasive', 'p'), ('Vasopressor (%)', 'vasopressor', 'p')]
ROWS = [r for r in ROWS if r[1] in df.columns]
def cell(s, kind): return miqr(s) if kind == 'c' else f'{s.mean()*100:.1f}'
char = {'Variable (0–24 h)': [r[0] for r in ROWS]}
for k in range(4):
    p = df[df.y == k]; char[f'{CLASSES[k]} (n={len(p):,})'] = [cell(p[c], t) for _, c, t in ROWS]
char[f'All (n={len(df):,})'] = [cell(df[c], t) for _, c, t in ROWS]
save_table(pd.DataFrame(char), 'table_3_cohort_characteristics')

trans = pd.crosstab(df['cat_0_24'], df['y']).reindex(index=range(4), columns=range(4)).fillna(0).astype(int)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), gridspec_kw={'width_ratios': [1, 1.15, 1]})
cnt = df['y'].value_counts().reindex(range(4)).fillna(0)
bars = axes[0].bar(CLASSES, cnt.values, color=CLASS_COLORS, width=0.6, edgecolor='white', linewidth=2)
for b, v in zip(bars, cnt.values):
    axes[0].text(b.get_x() + b.get_width()/2, v, f'{int(v):,}\n({v/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=8)
axes[0].set_ylim(0, cnt.max() * 1.2); axes[0].set_ylabel('ICU stays'); axes[0].set_title('(a) NEWS2 category at 24–48 h', loc='left')
pct = trans.div(trans.sum(axis=1).replace(0, 1), axis=0) * 100
ann = np.array([[f'{trans.iloc[i, j]:,}\n{pct.iloc[i, j]:.0f}%' for j in range(4)] for i in range(4)])
sns.heatmap(pct, annot=ann, fmt='', cmap='Blues', vmin=0, vmax=100, ax=axes[1], xticklabels=CLASSES, yticklabels=CLASSES,
            linewidths=1, linecolor='white', cbar_kws={'label': 'Row %'}, annot_kws={'size': 8})
axes[1].set_xlabel('Category at 24–48 h (outcome)'); axes[1].set_ylabel('Category at 0–24 h')
axes[1].set_title('(b) Transitions between windows', loc='left')
s = df.sample(n=min(20000, len(df)), random_state=SEED)
sns.boxplot(x=s['y'].map(dict(enumerate(CLASSES))), y=s['NEWS2_0_24'], order=CLASSES, palette=CLASS_COLORS,
            ax=axes[2], width=0.55, fliersize=0, linewidth=1)
axes[2].set_xlabel('Category at 24–48 h'); axes[2].set_ylabel('NEWS2 total at 0–24 h')
axes[2].set_title('(c) Early NEWS2 by later category', loc='left')
plt.tight_layout(); save_fig(fig, 'fig_2_cohort'); plt.show()


## 8. Model engine (shared with `tune_hyperparameters.ipynb`)
All learned models are fitted on the same 90% *fit* subset of the training patients; the remaining 10% (validation,
grouped by patient) is used for hyperparameter tuning, boosting early stopping, FT-Transformer checkpointing and the
choice of decision thresholds. The test patients are used only for the final evaluation.

**FT-Transformer** (Gorishniy et al., 2021): PLR periodic numerical embeddings (Gorishniy et al., 2022) on
quantile-normalised inputs, [CLS] token, pre-norm Transformer blocks with ReGLU feed-forward layers,
LayerNorm → ReLU → linear head; AdamW; class-weighted cross-entropy; early stopping on validation macro-F1.

In [ ]:
class LinearEmb(nn.Module):
    def __init__(s, n, d):
        super().__init__(); s.W = nn.Parameter(torch.empty(n, d)); s.b = nn.Parameter(torch.empty(n, d))
        for p in (s.W, s.b): nn.init.uniform_(p, -d ** -0.5, d ** -0.5)
    def forward(s, x): return x.unsqueeze(-1) * s.W + s.b
class PLREmb(nn.Module):                          # periodic → linear → ReLU, separate parameters per feature
    def __init__(s, n, d, k=48, sigma=0.01):
        super().__init__(); s.freq = nn.Parameter(torch.randn(n, k) * sigma)
        s.W = nn.Parameter(torch.empty(n, 2 * k, d)); s.b = nn.Parameter(torch.empty(n, d))
        for p in (s.W, s.b): nn.init.uniform_(p, -(2 * k) ** -0.5, (2 * k) ** -0.5)
    def forward(s, x):
        v = 2 * math.pi * s.freq[None] * x[..., None]
        v = torch.cat([torch.cos(v), torch.sin(v)], -1)
        return torch.relu(torch.einsum('bnk,nkd->bnd', v, s.W) + s.b)
class ReGLU(nn.Module):
    def forward(s, x): a, b = x.chunk(2, -1); return a * torch.relu(b)
class Block(nn.Module):                           # pre-norm block; the first block has no attention pre-norm (reference)
    def __init__(s, d, heads, ffn, att_drop, ffn_drop, res_drop, first):
        super().__init__(); s.n1 = nn.Identity() if first else nn.LayerNorm(d)
        s.att = nn.MultiheadAttention(d, heads, dropout=att_drop, batch_first=True)
        s.n2 = nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, 2 * ffn), ReGLU(), nn.Dropout(ffn_drop), nn.Linear(ffn, d))
        s.rd = nn.Dropout(res_drop)
    def forward(s, x):
        h = s.n1(x); x = x + s.rd(s.att(h, h, h, need_weights=False)[0])
        return x + s.rd(s.ff(s.n2(x)))
class FTTransformer(nn.Module):
    def __init__(s, n, k, d=192, layers=3, ffn=256, att_drop=0.2, ffn_drop=0.1, res_drop=0.0, n_freq=48, sigma=0.01, heads=8, emb='plr'):
        super().__init__()
        s.tok = PLREmb(n, d, n_freq, sigma) if emb == 'plr' else LinearEmb(n, d)
        s.cls = nn.Parameter(torch.empty(1, 1, d)); nn.init.uniform_(s.cls, -d ** -0.5, d ** -0.5)
        s.blocks = nn.ModuleList([Block(d, heads, ffn, att_drop, ffn_drop, res_drop, i == 0) for i in range(layers)])
        s.head = nn.Sequential(nn.LayerNorm(d), nn.ReLU(), nn.Linear(d, k))
    def forward(s, x):
        t = torch.cat([s.cls.expand(x.shape[0], -1, -1), s.tok(x)], 1)
        for b in s.blocks: t = b(t)
        return s.head(t[:, 0])

FTT_KEYS = ['d', 'layers', 'ffn', 'att_drop', 'ffn_drop', 'res_drop', 'n_freq', 'sigma']
T = lambda a, dt=torch.float32: torch.as_tensor(np.asarray(a), dtype=dt).to(device)
@torch.no_grad()
def proba_of(net, X, bs=8192):
    net.eval(); return np.concatenate([torch.softmax(net(T(X[i:i+bs])), 1).cpu().numpy() for i in range(0, len(X), bs)])
def ens_proba(nets, X): return np.mean([proba_of(m, X) for m in nets], 0)

def train_ftt(Xf, yf, Xv, yv, K, P, class_w, seed, max_epochs=None, patience=None, trial=None, log_every=10):
    max_epochs, patience = max_epochs or FTT_EPOCHS, patience or FTT_PATIENCE
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    net = FTTransformer(Xf.shape[1], K, **{k: P[k] for k in FTT_KEYS}, emb=FTT_EMB).to(device)
    opt = optim.AdamW(net.parameters(), lr=P['lr'], weight_decay=P['wd'])
    crit = nn.CrossEntropyLoss(weight=T(class_w) if class_w is not None else None)
    dl = DataLoader(TensorDataset(T(Xf), T(yf, torch.long)), batch_size=256, shuffle=True,
                    generator=torch.Generator().manual_seed(seed))
    best, wait, hist = (-1, None, 0), 0, []
    for ep in range(1, max_epochs + 1):
        net.train(); tot = 0
        for xb, yb in dl:
            opt.zero_grad(); loss = crit(net(xb), yb); loss.backward(); opt.step(); tot += loss.item()
        vf1 = f1_score(yv, proba_of(net, Xv).argmax(1), average='macro')
        hist.append({'seed': seed, 'epoch': ep, 'train_loss': tot / len(dl), 'val_macro_f1': vf1})
        if vf1 > best[0]: best, wait = (vf1, copy.deepcopy(net.state_dict()), ep), 0
        else: wait += 1
        if log_every and ep % log_every == 0: print(f'    seed {seed} epoch {ep:3d} | loss {tot/len(dl):.4f} | val macro-F1 {vf1:.4f}')
        if trial is not None:
            trial.report(vf1, ep)
            if trial.should_prune(): raise optuna.TrialPruned()
        if wait >= patience: break
    net.load_state_dict(best[1]); net.eval()
    return net, best[2], best[0], hist, ep

def xgb_device():
    import xgboost
    if not torch.cuda.is_available(): return {'tree_method': 'hist'}
    return {'tree_method': 'hist', 'device': 'cuda'} if int(xgboost.__version__.split('.')[0]) >= 2 else {'tree_method': 'gpu_hist'}
XGB_FIXED = dict(n_estimators=3000, eval_metric='mlogloss', early_stopping_rounds=100, random_state=SEED, verbosity=0, n_jobs=-1, **xgb_device())
LGBM_FIXED = dict(n_estimators=3000, subsample_freq=1, random_state=SEED, verbose=-1)
def make_models(P, weighted=True):
    bal = 'balanced' if weighted else None
    return {'Logistic regression': LogisticRegression(max_iter=3000, class_weight=bal, random_state=SEED),
            'Random forest': RandomForestClassifier(**P['rf'], class_weight=bal, random_state=SEED, n_jobs=-1),
            'XGBoost': XGBClassifier(**P['xgb'], **XGB_FIXED),
            'LightGBM': LGBMClassifier(**P['lgbm'], **LGBM_FIXED, class_weight=bal)}
def fit_model(name, m, Xs_tr, ytr, fit, val, weighted=True):
    if name == 'XGBoost':
        sw = compute_sample_weight('balanced', ytr[fit]) if weighted else None
        m.fit(Xs_tr[fit], ytr[fit], sample_weight=sw, eval_set=[(Xs_tr[val], ytr[val])], verbose=False)
        return int(m.best_iteration) + 1
    if name == 'LightGBM':
        m.fit(Xs_tr[fit], ytr[fit], eval_set=[(Xs_tr[val], ytr[val])], callbacks=[lgbm.early_stopping(100, verbose=False)])
        return int(m.best_iteration_)
    m.fit(Xs_tr[fit], ytr[fit]); return None

def evaluate(name, y, pred, proba=None, K=4):
    try: au = roc_auc_score(y, proba, multi_class='ovr', average='macro') if proba is not None else np.nan
    except ValueError: au = np.nan
    return {'Model': name, 'Accuracy': accuracy_score(y, pred), 'Balanced Acc': balanced_accuracy_score(y, pred),
            'Macro-F1': f1_score(y, pred, average='macro', zero_division=0),
            'Weighted-F1': f1_score(y, pred, average='weighted', zero_division=0), 'Macro-AUROC': au,
            'High recall': recall_score(y, pred, labels=[K - 1], average='macro', zero_division=0),
            'Brier': np.mean(np.sum((proba - np.eye(K)[y]) ** 2, axis=1)) if proba is not None else np.nan}

def run_models(Xs_tr, Xs_te, Xq_tr, Xq_te, ytr, yte, K, weighted, n_seeds, fit, val, pers=None, verbose=True):
    # Fit every comparator and the FT-Transformer seed ensemble on one task; returns predictions and metrics.
    R = {'store': {}, 'val': {}, 'results': [], 'models': {}, 'best_iter': {}, 'hist': [], 'seeds': []}
    cwv = compute_class_weight('balanced', classes=np.arange(K), y=ytr[fit]) if weighted else None
    if pers is not None:
        R['store']['NEWS2 persistence'] = (pers, None); R['results'].append(evaluate('NEWS2 persistence', yte, pers, None, K))
    for name, m in make_models(PARAMS, weighted).items():
        t0 = time.time(); it = fit_model(name, m, Xs_tr, ytr, fit, val, weighted)
        pr = m.predict_proba(Xs_te); R['store'][name] = (pr.argmax(1), pr); R['val'][name] = m.predict_proba(Xs_tr[val])
        R['models'][name] = m; R['results'].append(evaluate(name, yte, pr.argmax(1), pr, K))
        if it is not None: R['best_iter'][name] = it
        if verbose: print(f"  ▶ {name:20s} macro-F1 {R['results'][-1]['Macro-F1']:.4f} | AUROC {R['results'][-1]['Macro-AUROC']:.4f} | "
                          f"accuracy {R['results'][-1]['Accuracy']:.4f}" + (f' | trees {it}' if it else '') + f' | {time.time()-t0:.0f}s')
    nets = []
    for si in range(n_seeds):
        t0 = time.time()
        net, bep, bf1, h, ep = train_ftt(Xq_tr[fit], ytr[fit], Xq_tr[val], ytr[val], K, PARAMS['ftt'], cwv, SEED + si,
                                         log_every=10 if verbose else 0)
        nets.append(net); R['hist'] += h; pt = proba_of(net, Xq_te)
        R['seeds'].append({'seed': SEED + si, 'best epoch': bep, 'epochs run': ep, 'val macro-F1': bf1,
                           'test macro-F1 (this seed)': f1_score(yte, pt.argmax(1), average='macro'),
                           'test accuracy (this seed)': accuracy_score(yte, pt.argmax(1)),
                           'minutes': (time.time() - t0) / 60})
        if not verbose: print(f'    FT-Transformer seed {SEED+si} done: best epoch {bep} of {ep} | {(time.time()-t0)/60:.1f} min')
        if verbose: print(f"  ▶ FT-Transformer seed {SEED+si}: best epoch {bep} of {ep} | val macro-F1 {bf1:.4f} | "
                          f"test macro-F1 {R['seeds'][-1]['test macro-F1 (this seed)']:.4f} | {R['seeds'][-1]['minutes']:.1f} min")
    pr = ens_proba(nets, Xq_te); R['store']['FT-Transformer'] = (pr.argmax(1), pr); R['val']['FT-Transformer'] = ens_proba(nets, Xq_tr[val])
    R['nets'] = nets; R['n_params'] = sum(p.numel() for p in nets[0].parameters())
    R['results'].append(evaluate('FT-Transformer', yte, pr.argmax(1), pr, K))
    R['res'] = pd.DataFrame(R['results']).sort_values('Macro-F1', ascending=False).reset_index(drop=True)
    return R

# Published default settings (used when no tuning output is attached)
DEFAULT_PARAMS = {
    'ftt':  dict(d=192, layers=3, ffn=256, att_drop=0.2, ffn_drop=0.1, res_drop=0.0, n_freq=48, sigma=0.01, lr=1e-4, wd=1e-5),
    'xgb':  dict(max_depth=6, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, min_child_weight=5),
    'lgbm': dict(num_leaves=63, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8, min_child_samples=40),
    'rf':   dict(n_estimators=300, max_depth=14, min_samples_leaf=5, max_features='sqrt')}
PARAMS = copy.deepcopy(DEFAULT_PARAMS)
print('engine ready | XGBoost:', xgb_device())


## 8b. Hyperparameters
Loads `best_params.json` written by `tune_hyperparameters.ipynb` (attach that notebook's output as an input).
If it is absent, the published default settings are used and the run is labelled accordingly.

In [ ]:
import json
P_BEST = find('best_params.json')
PARAM_SOURCE = 'published defaults (no tuning output attached)'
if P_BEST:
    tuned = json.load(open(P_BEST))
    if tuned.get('features') != FEATURES or tuned.get('n_fit') != len(fit_i):
        print('⚠ best_params.json was made with a different feature set or split — check that both notebooks use the same inputs.')
    for k in ['ftt', 'xgb', 'lgbm', 'rf']:
        if k in tuned: PARAMS[k] = tuned[k]
    PARAM_SOURCE = f"Optuna ({', '.join(f'{k}: {v} trials' for k, v in tuned.get('n_trials', {}).items())})"
print('hyperparameter source:', PARAM_SOURCE)
for k, v in PARAMS.items(): print(f'  {k:5s}', v)


## 9. Primary analysis (A): four NEWS2 categories at 24–48 h, class-weighted
NEWS2 persistence, logistic regression, random forest, XGBoost, LightGBM and the FT-Transformer
(`N_SEEDS`-seed ensemble), all with the hyperparameters loaded above.

In [ ]:
R = run_models(X_tr_s, X_te_s, X_tr_q, X_te_q, y_tr, y_te, 4, True, N_SEEDS, fit_i, val_i, pers=te['cat_0_24'].values)
store, VAL, models, BEST_ITER, FTT_MODELS, n_params = R['store'], R['val'], R['models'], R['best_iter'], R['nets'], R['n_params']
results, res, hist, seeds_df = R['results'], R['res'], R['hist'], pd.DataFrame(R['seeds'])
def ftt_proba(Xq): return ens_proba(FTT_MODELS, Xq)
print(f"FT-Transformer: {n_params:,} parameters per model | ensemble of {N_SEEDS} | single-seed test macro-F1 "
      f"{seeds_df['test macro-F1 (this seed)'].mean():.4f} ± {seeds_df['test macro-F1 (this seed)'].std():.4f}")
save_table(pd.DataFrame(hist), 'table_s1_ftt_training_history'); save_table(seeds_df.round(4), 'table_s3_ftt_seeds')
print(res.round(4).to_string(index=False))


## 10. Patient-clustered bootstrap: 95% CIs and paired differences

In [ ]:
rng = np.random.default_rng(SEED)
pt_idx = pd.Series(np.arange(len(te))).groupby(te['subject_id'].values).apply(np.array)
BOOT = [np.concatenate(pt_idx.iloc[rng.integers(0, len(pt_idx), len(pt_idx))].values) for _ in range(N_BOOT)]
MET = ['Macro-F1', 'Balanced Acc', 'Macro-AUROC', 'High recall']
def boot(pred, proba):
    out = {m: [] for m in MET}
    for ix in BOOT:
        yt, yp = y_te[ix], pred[ix]
        out['Macro-F1'].append(f1_score(yt, yp, average='macro', zero_division=0))
        out['Balanced Acc'].append(balanced_accuracy_score(yt, yp))
        out['High recall'].append(recall_score(yt, yp, labels=[3], average='macro', zero_division=0))
        if proba is not None:
            try: out['Macro-AUROC'].append(roc_auc_score(yt, proba[ix], multi_class='ovr', average='macro'))
            except ValueError: pass
    return {k: np.array(v) for k, v in out.items()}
B = {}
for name, (pred_, proba_) in store.items():
    print('bootstrapping', name); B[name] = boot(pred_, proba_)
for m in MET:
    res[f'{m} lo'] = [np.percentile(B[n][m], 2.5) if len(B[n][m]) else np.nan for n in res.Model]
    res[f'{m} hi'] = [np.percentile(B[n][m], 97.5) if len(B[n][m]) else np.nan for n in res.Model]
tbl = res[['Model']].copy(); tbl['Accuracy'] = res['Accuracy'].round(3)
for m in MET:
    tbl[f'{m} (95% CI)'] = [('—' if np.isnan(r[m]) else f"{r[m]:.3f} ({r[m+' lo']:.3f}–{r[m+' hi']:.3f})") for _, r in res.iterrows()]
tbl['Weighted-F1'] = res['Weighted-F1'].round(3); tbl['Brier'] = res['Brier'].round(4)
save_table(tbl, 'table_4_performance'); print(tbl.to_string(index=False))

diffs = []
for other in [n for n in store if n != 'FT-Transformer']:
    for m in ['Macro-F1', 'Macro-AUROC', 'High recall']:
        if not len(B[other][m]): continue
        d = B['FT-Transformer'][m] - B[other][m]
        diffs.append({'Comparison': f'FT-Transformer − {other}', 'Metric': m, 'Mean difference': d.mean(),
                      'CI lo': np.percentile(d, 2.5), 'CI hi': np.percentile(d, 97.5), 'P(diff ≤ 0)': (d <= 0).mean()})
diff_df = pd.DataFrame(diffs); save_table(diff_df.round(4), 'table_5_paired_differences')
print(diff_df.round(4).to_string(index=False))


## 11. Per-class results, confusion matrices and error distance

In [ ]:
BEST_BASE = res[~res.Model.isin(['FT-Transformer', 'NEWS2 persistence'])].iloc[0]['Model']
rows, err = [], []
for name in ['FT-Transformer', BEST_BASE, 'NEWS2 persistence']:
    yp = store[name][0]
    p = precision_score(y_te, yp, labels=range(4), average=None, zero_division=0)
    r = recall_score(y_te, yp, labels=range(4), average=None, zero_division=0)
    f = f1_score(y_te, yp, labels=range(4), average=None, zero_division=0)
    for k in range(4):
        rows.append({'Model': name, 'Category': CLASSES[k], 'Precision': p[k], 'Recall': r[k], 'F1': f[k], 'Support': int((y_te == k).sum())})
    cm = confusion_matrix(y_te, yp, labels=range(4))
    save_table(pd.DataFrame(cm, index=[f'True {c}' for c in CLASSES], columns=[f'Pred {c}' for c in CLASSES]).reset_index(),
               'table_cm_' + ''.join(ch if ch.isalnum() else '_' for ch in name.lower()))
    d = np.abs(y_te - yp)[y_te != yp]; n = max(len(d), 1)
    err.append({'Model': name, 'Errors': len(d), 'Error rate %': len(d) / len(y_te) * 100,
                'One category away %': (d == 1).sum() / n * 100, 'Two away %': (d == 2).sum() / n * 100,
                'Low↔High %': (d == 3).sum() / n * 100, 'High predicted Low/Low–medium (n)': int(((y_te == 3) & (yp <= 1)).sum())})
save_table(pd.DataFrame(rows).round(4), 'table_6_per_class')
err_df = pd.DataFrame(err); save_table(err_df.round(2), 'table_7_error_distance'); print(err_df.round(2).to_string(index=False))


## 11b. Secondary outcome: High risk or death at 24–48 h (binary)
Score = predicted probability of the High category from each four-class model; the clinical comparator is the
NEWS2 total at 0–24 h. AUROC and AUPRC with 95% patient-clustered bootstrap CIs. Accuracy, sensitivity, specificity,
PPV, NPV and F1 at thresholds chosen on the validation patients (Youden; ≥80% sensitivity), plus the NEWS2 clinical
trigger levels (≥5, ≥7). Reference: accuracy of predicting nobody High.

In [ ]:
yb = (y_te == 3).astype(int)
SCORES = {n: store[n][1][:, 3] for n in store if store[n][1] is not None}
SCORES['NEWS2 score (0–24 h)'] = te['NEWS2_0_24'].values.astype(float)
BB, rows = {}, []
for n, s in SCORES.items():
    au_b, ap_b = [], []
    for ix in BOOT:
        if yb[ix].min() == yb[ix].max(): continue
        au_b.append(roc_auc_score(yb[ix], s[ix])); ap_b.append(average_precision_score(yb[ix], s[ix]))
    BB[n] = (np.array(au_b), np.array(ap_b))
    au, ap = roc_auc_score(yb, s), average_precision_score(yb, s)
    rows.append({'Model': n, 'AUROC': au, 'AUROC lo': np.percentile(au_b, 2.5), 'AUROC hi': np.percentile(au_b, 97.5),
                 'AUPRC': ap, 'AUPRC lo': np.percentile(ap_b, 2.5), 'AUPRC hi': np.percentile(ap_b, 97.5)})
bin_df = pd.DataFrame(rows).sort_values('AUROC', ascending=False).reset_index(drop=True)
bin_tbl = pd.DataFrame({'Model': bin_df.Model,
    'AUROC (95% CI)': [f"{r.AUROC:.3f} ({r['AUROC lo']:.3f}–{r['AUROC hi']:.3f})" for _, r in bin_df.iterrows()],
    'AUPRC (95% CI)': [f"{r.AUPRC:.3f} ({r['AUPRC lo']:.3f}–{r['AUPRC hi']:.3f})" for _, r in bin_df.iterrows()]})
bin_tbl['Prevalence'] = round(yb.mean(), 3)
save_table(bin_tbl, 'table_10_high_risk_binary'); print(bin_tbl.to_string(index=False))
bd = []
for other in [n for n in SCORES if n != 'FT-Transformer']:
    for k, m in enumerate(['AUROC', 'AUPRC']):
        d = BB['FT-Transformer'][k] - BB[other][k]
        bd.append({'Comparison': f'FT-Transformer − {other}', 'Metric': m, 'Mean difference': d.mean(),
                   'CI lo': np.percentile(d, 2.5), 'CI hi': np.percentile(d, 97.5)})
save_table(pd.DataFrame(bd).round(4), 'table_11_high_risk_differences')

# Classification at thresholds chosen on the validation patients (never on test):
#   Youden's J (max sensitivity + specificity − 1) and the highest threshold reaching ≥80% sensitivity.
#   NEWS2 score: Youden, ≥80% sensitivity, and the RCP clinical trigger levels (≥5 urgent response, ≥7 emergency response).
yvb = (y_tr[val_i] == 3).astype(int)
VSC = {n: VAL[n][:, 3] for n in VAL}; VSC['NEWS2 score (0–24 h)'] = tr['NEWS2_0_24'].values[val_i].astype(float)
def thr_youden(y, s): f, t, th = roc_curve(y, s); return th[np.argmax(t - f)]
def thr_sens(y, s, target=0.8): f, t, th = roc_curve(y, s); return th[np.argmax(t >= target)]
def cls_metrics(y, p):
    tp, tn = ((y == 1) & (p == 1)).sum(), ((y == 0) & (p == 0)).sum(); fp, fn = ((y == 0) & (p == 1)).sum(), ((y == 1) & (p == 0)).sum()
    sd = lambda a, b: a / b if b else np.nan
    return {'Accuracy': (tp + tn) / len(y), 'Sensitivity': sd(tp, tp + fn), 'Specificity': sd(tn, tn + fp),
            'PPV': sd(tp, tp + fp), 'NPV': sd(tn, tn + fn), 'F1': sd(2 * tp, 2 * tp + fp + fn)}
RULES = []
for n in SCORES:
    RULES += [(n, 'Youden (validation)', thr_youden(yvb, VSC[n])), (n, '≥80% sensitivity (validation)', thr_sens(yvb, VSC[n]))]
RULES += [('NEWS2 score (0–24 h)', 'NEWS2 ≥5 (clinical trigger)', 5), ('NEWS2 score (0–24 h)', 'NEWS2 ≥7 (clinical trigger)', 7)]
CM_KEYS = ['Accuracy', 'Sensitivity', 'Specificity', 'PPV', 'NPV', 'F1']
thr_rows, OPS = [], {}
for n, rule, th in RULES:
    p = (SCORES[n] >= th).astype(int); pt = cls_metrics(yb, p)
    bs = pd.DataFrame([cls_metrics(yb[ix], p[ix]) for ix in BOOT])
    row = {'Model': n, 'Rule': rule, 'Threshold': round(float(th), 4)}
    for k in CM_KEYS: row[k] = pt[k]; row[k + ' lo'] = bs[k].quantile(.025); row[k + ' hi'] = bs[k].quantile(.975)
    thr_rows.append(row); OPS[(n, rule)] = (1 - pt['Specificity'], pt['Sensitivity'])
thr_df = pd.DataFrame(thr_rows)
thr_tbl = thr_df[['Model', 'Rule', 'Threshold']].copy()
for k in CM_KEYS: thr_tbl[f'{k} (95% CI)'] = [f"{r[k]:.3f} ({r[k+' lo']:.3f}–{r[k+' hi']:.3f})" for _, r in thr_df.iterrows()]
thr_tbl.loc[len(thr_tbl)] = ['Reference: predict nobody High', '—', '—', f'{1 - yb.mean():.3f}'] + ['—'] * (len(CM_KEYS) - 1)
save_table(thr_tbl, 'table_12_high_risk_thresholds'); print(thr_tbl.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for n, c in [('FT-Transformer', C_ORANGE), (BEST_BASE, C_AQUA), ('NEWS2 score (0–24 h)', C_GREY)]:
    fpr, tpr, _ = roc_curve(yb, SCORES[n]); pr_, rc_, _ = precision_recall_curve(yb, SCORES[n])
    axes[0].plot(fpr, tpr, color=c, lw=2, label=f'{n} (AUROC {roc_auc_score(yb, SCORES[n]):.3f})')
    axes[1].plot(rc_, pr_, color=c, lw=2, label=f'{n} (AUPRC {average_precision_score(yb, SCORES[n]):.3f})')
for (n, rule), (fx, ty) in OPS.items():
    if n == 'FT-Transformer' or (n.startswith('NEWS2') and 'clinical' in rule):
        mk = {'Youden (validation)': 'o', '≥80% sensitivity (validation)': 's'}.get(rule, 'D')
        axes[0].scatter(fx, ty, marker=mk, s=45, color=C_ORANGE if n == 'FT-Transformer' else C_GREY, edgecolor='white', zorder=4)
axes[0].plot([0, 1], [0, 1], color='#c3c2b7', ls='--', lw=1); axes[1].axhline(yb.mean(), color='#c3c2b7', ls='--', lw=1)
axes[0].text(0.98, 0.32, 'Markers: ● Youden  ■ ≥80% sensitivity\n◆ NEWS2 ≥5 / ≥7', transform=axes[0].transAxes, ha='right', fontsize=7.5, color='#52514e')
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate'); axes[0].set_title('(a) ROC: High risk or death at 24–48 h', loc='left')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('(b) Precision–recall (dashed: prevalence)', loc='left')
for a in axes: a.legend(loc='lower right' if a is axes[0] else 'upper right', fontsize=8)
plt.tight_layout(); save_fig(fig, 'fig_7_high_risk'); plt.show()


## 12. Performance figures

In [ ]:
def col_for(m): return C_ORANGE if m == 'FT-Transformer' else C_GREY if m.startswith('NEWS2') else C_AQUA
r = res.sort_values('Macro-F1').reset_index(drop=True)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), sharey=True)
for ax, m, ttl in zip(axes, ['Macro-F1', 'Balanced Acc', 'Macro-AUROC'], ['(a) Macro-F1', '(b) Balanced accuracy', '(c) Macro-AUROC (one-vs-rest)']):
    v, lo, hi = r[m].values, r[f'{m} lo'].values, r[f'{m} hi'].values
    bars = ax.barh(r['Model'], np.nan_to_num(v), color=[col_for(x) for x in r['Model']], height=0.62, edgecolor='white', linewidth=2)
    ok = ~np.isnan(v) & ~np.isnan(lo)
    ax.errorbar(v[ok], np.arange(len(r))[ok], xerr=[np.clip(v[ok] - lo[ok], 0, None), np.clip(hi[ok] - v[ok], 0, None)], fmt='none', ecolor='#0b0b0b', elinewidth=1, capsize=3)
    for i, b in enumerate(bars):
        ax.text((hi[i] if ok[i] else np.nan_to_num(v[i])) + 0.015, b.get_y() + b.get_height()/2,
                '—' if np.isnan(v[i]) else f'{v[i]:.3f}', va='center', fontsize=8)
    ax.set_xlim(0, 1.1); ax.set_xlabel(m); ax.set_title(ttl, loc='left'); ax.grid(axis='x', color='#e6e5e1'); ax.set_axisbelow(True)
fig.legend(handles=[mpatches.Patch(color=C_ORANGE, label='FT-Transformer (proposed)'), mpatches.Patch(color=C_AQUA, label='Comparator models'),
                    mpatches.Patch(color=C_GREY, label='NEWS2 persistence baseline')], loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.07))
fig.text(0.5, -0.12, 'Error bars: 95% patient-clustered bootstrap confidence intervals.', ha='center', fontsize=8, color='#52514e')
plt.tight_layout(); save_fig(fig, 'fig_3_performance'); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, name, lab in zip(axes, ['FT-Transformer', BEST_BASE, 'NEWS2 persistence'], 'abc'):
    cm = confusion_matrix(y_te, store[name][0], labels=range(4)); pc = cm / np.maximum(cm.sum(1, keepdims=True), 1) * 100
    ann = np.array([[f'{cm[i,j]:,}\n{pc[i,j]:.0f}%' for j in range(4)] for i in range(4)])
    sns.heatmap(pc, annot=ann, fmt='', cmap='Blues', vmin=0, vmax=100, ax=ax, cbar=ax is axes[-1], xticklabels=CLASSES,
                yticklabels=CLASSES, linewidths=1, linecolor='white', annot_kws={'size': 8}, cbar_kws={'label': 'Row % (recall)'})
    ax.set_title(f'({lab}) {name}', loc='left'); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); save_fig(fig, 'fig_4_confusion'); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, name, lab in zip(axes, ['FT-Transformer', BEST_BASE], 'ab'):
    pr_ = store[name][1]
    for k, c in enumerate(CLASS_COLORS):
        if (y_te == k).sum() == 0: continue
        fpr, tpr, _ = roc_curve((y_te == k).astype(int), pr_[:, k])
        ax.plot(fpr, tpr, color=c, lw=2, label=f'{CLASSES[k]} (AUC {auc(fpr, tpr):.3f})')
    ax.plot([0, 1], [0, 1], color=C_GREY, lw=1, ls='--'); ax.set_xlabel('False positive rate')
    ax.set_title(f'({lab}) {name}', loc='left'); ax.legend(loc='lower right', fontsize=8)
axes[0].set_ylabel('True positive rate')
plt.tight_layout(); save_fig(fig, 'fig_5_roc'); plt.show()


## 13. Explainability
**FT-Transformer:** permutation importance — drop in macro-AUROC on held-out patients when a feature (or all
features of one group together) is shuffled, averaged over `N_PERM` repeats.
**LightGBM:** mean |SHAP| (TreeExplainer) as an independent cross-check. Agreement: Spearman ρ between rankings.

In [ ]:
rng = np.random.default_rng(SEED)
sub = rng.choice(len(X_te_q), min(5000, len(X_te_q)), replace=False)
Xs, ys = X_te_q[sub], y_te[sub]                  # FT-Transformer input space
base_auc = roc_auc_score(ys, ftt_proba(Xs), multi_class='ovr', average='macro')
def perm_drop(cols):
    drops = []
    for _ in range(N_PERM):
        Xp = Xs.copy(); idx = rng.permutation(len(Xp)); Xp[:, cols] = Xp[idx][:, cols]
        drops.append(base_auc - roc_auc_score(ys, ftt_proba(Xp), multi_class='ovr', average='macro'))
    return np.mean(drops), np.std(drops)
feat_imp = pd.DataFrame([{'Feature': f, 'Name': nice(f), 'Group': GROUP[f], **dict(zip(['AUROC drop', 'SD'], perm_drop([i])))}
                         for i, f in enumerate(FEATURES)])
groups = sorted(set(GROUP.values()), key=lambda g: list(GROUP.values()).index(g))
grp_imp = pd.DataFrame([{'Group': g, **dict(zip(['AUROC drop', 'SD'], perm_drop([i for i, f in enumerate(FEATURES) if GROUP[f] == g])))}
                        for g in groups]).sort_values('AUROC drop', ascending=False)

lgb = models['LightGBM']
sv = shap.TreeExplainer(lgb).shap_values(X_te_s[sub[:1000]], check_additivity=False)
sv = np.mean([np.abs(s).mean(0) for s in sv], 0) if isinstance(sv, list) else (np.abs(sv).mean((0, 2)) if np.ndim(sv) == 3 else np.abs(sv).mean(0))
feat_imp['LightGBM mean|SHAP|'] = sv
feat_imp['Rank FT-T'] = feat_imp['AUROC drop'].rank(ascending=False, method='min').astype(int)
feat_imp['Rank LightGBM'] = feat_imp['LightGBM mean|SHAP|'].rank(ascending=False, method='min').astype(int)
rho, pval = spearmanr(feat_imp['AUROC drop'], feat_imp['LightGBM mean|SHAP|'])
feat_imp = feat_imp.sort_values('AUROC drop', ascending=False).reset_index(drop=True)
save_table(feat_imp.round(5), 'table_8_feature_importance'); save_table(grp_imp.round(5), 'table_9_vital_sign_importance')
print(grp_imp.round(4).to_string(index=False)); print(f'Spearman ρ (FT-T permutation vs LightGBM SHAP) = {rho:.3f}, p = {pval:.3g}')

DCOL = {'Vital signs': '#2a78d6', 'Neurological & O₂': '#eb6834', 'Laboratory': '#1baf7a', 'Treatment': '#eda100',
        'Context': '#e87ba4', 'NEWS2': '#9a9893'}
GDOM = {g: DOMAIN[[f for f in FEATURES if GROUP[f] == g][0]] for g in set(GROUP.values())}
GNAME = {'SpO2': 'SpO₂', 'Temp': 'Temperature'}
fig, axes = plt.subplots(1, 3, figsize=(18, 6.2), gridspec_kw={'width_ratios': [0.8, 1, 1]})
g = grp_imp.iloc[::-1]
axes[0].barh([GNAME.get(x, x) for x in g['Group']], g['AUROC drop'], xerr=g['SD'], color=[DCOL[GDOM[x]] for x in g['Group']],
             height=0.6, edgecolor='white', linewidth=1.5, error_kw={'elinewidth': 1, 'capsize': 3})
axes[0].set_xlabel('Drop in macro-AUROC when shuffled'); axes[0].set_title('(a) FT-Transformer: importance by feature group', loc='left')
top = feat_imp.head(15).iloc[::-1]
axes[1].barh(top['Name'], top['AUROC drop'], color=[DCOL[DOMAIN[f]] for f in top['Feature']], height=0.65, edgecolor='white', linewidth=1.5)
axes[1].set_xlabel('Drop in macro-AUROC when shuffled'); axes[1].set_title('(b) FT-Transformer: top 15 features', loc='left')
for _, row in feat_imp.iterrows():
    axes[2].scatter(row['Rank LightGBM'], row['Rank FT-T'], s=50, color=DCOL[DOMAIN[row['Feature']]], edgecolor='white', linewidth=1.2, zorder=3)
for _, row in feat_imp.head(5).iterrows():
    axes[2].annotate(row['Name'], (row['Rank LightGBM'], row['Rank FT-T']), textcoords='offset points', xytext=(5, 3), fontsize=7, color='#52514e')
n = len(FEATURES); axes[2].plot([1, n], [1, n], color=C_GREY, ls='--', lw=1); axes[2].invert_yaxis()
axes[2].set_xlabel('Rank by LightGBM mean |SHAP|'); axes[2].set_ylabel('Rank by FT-Transformer permutation importance')
axes[2].set_title(f'(c) Agreement between models (Spearman ρ = {rho:.2f})', loc='left')
for a in axes[:2]: a.grid(axis='x', color='#e6e5e1'); a.set_axisbelow(True)
fig.legend(handles=[mpatches.Patch(color=c, label=k) for k, c in DCOL.items() if k in set(DOMAIN.values())], loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.06), fontsize=8)
plt.tight_layout(); save_fig(fig, 'fig_6_explainability'); plt.show()


## 13b. Sensitivity analyses (C–E)
Same features, split and hyperparameters; all models retrained (FT-Transformer: `N_SEEDS_SENS` seeds).
**C** — no class weighting (maximises accuracy at the expense of rare categories).
**D** — three categories: Low (Low + Low–medium) / Medium / High.
**E** — outcome window shortened to 24–30 h (stays with ≥3 core vitals at 24–30 h or death in 24–30 h).
95% patient-clustered bootstrap CIs.

In [ ]:
def boot_ci(yt, pred, proba, subj, K, n=N_BOOT):
    rng_ = np.random.default_rng(SEED)
    pidx = pd.Series(np.arange(len(yt))).groupby(subj).apply(np.array)
    out = {'Accuracy': [], 'Macro-F1': [], 'Balanced Acc': [], 'Macro-AUROC': []}
    for _ in range(n):
        ix = np.concatenate(pidx.iloc[rng_.integers(0, len(pidx), len(pidx))].values)
        out['Accuracy'].append(accuracy_score(yt[ix], pred[ix])); out['Macro-F1'].append(f1_score(yt[ix], pred[ix], average='macro', zero_division=0))
        out['Balanced Acc'].append(balanced_accuracy_score(yt[ix], pred[ix]))
        if proba is not None:
            try: out['Macro-AUROC'].append(roc_auc_score(yt[ix], proba[ix], multi_class='ovr', average='macro'))
            except ValueError: pass
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) if len(v) else (np.nan, np.nan) for k, v in out.items()}
SENS_MET = ['Accuracy', 'Macro-F1', 'Balanced Acc', 'Macro-AUROC']
sens_rows = []
def add_rows(label, Rr, yt, subj, K):
    for _, r in Rr['res'].iterrows():
        pred_, proba_ = Rr['store'][r.Model]; ci = boot_ci(yt, pred_, proba_, subj, K)
        row = {'Analysis': label, 'Model': r.Model, 'Test stays': len(yt)}
        for m in SENS_MET: row[m] = r[m]; row[m + ' lo'], row[m + ' hi'] = ci[m]
        sens_rows.append(row)
add_rows('A: primary (4 categories, 24–48 h)', R, y_te, te['subject_id'].values, 4)
SENS_RUNS = {}
if RUN_SENS:
    print('▶ C: no class weighting'); t0 = time.time()
    SENS_RUNS['C'] = Rc = run_models(X_tr_s, X_te_s, X_tr_q, X_te_q, y_tr, y_te, 4, False, N_SEEDS_SENS, fit_i, val_i, pers=te['cat_0_24'].values, verbose=False)
    add_rows('C: no class weighting', Rc, y_te, te['subject_id'].values, 4); print(f'  done in {(time.time()-t0)/60:.1f} min')

    print('▶ D: three categories'); t0 = time.time(); M3 = np.array([0, 0, 1, 2])
    SENS_RUNS['D'] = Rd = run_models(X_tr_s, X_te_s, X_tr_q, X_te_q, M3[y_tr], M3[y_te], 3, True, N_SEEDS_SENS, fit_i, val_i,
                                     pers=M3[te['cat_0_24'].values.astype(int)], verbose=False)
    add_rows('D: three categories', Rd, M3[y_te], te['subject_id'].values, 3); print(f'  done in {(time.time()-t0)/60:.1f} min')

    print('▶ E: 24–30 h outcome'); t0 = time.time()
    mtr, mte = tr['y_6h'].notnull().values, te['y_6h'].notnull().values
    val_subj = set(tr['subject_id'].values[val_i]); s_tr = tr['subject_id'].values[mtr]
    fit6, val6 = np.where(~np.isin(s_tr, list(val_subj)))[0], np.where(np.isin(s_tr, list(val_subj)))[0]
    y6_tr, y6_te = tr['y_6h'].values[mtr].astype(int), te['y_6h'].values[mte].astype(int)
    SENS_RUNS['E'] = Re = run_models(X_tr_s[mtr], X_te_s[mte], X_tr_q[mtr], X_te_q[mte], y6_tr, y6_te, 4, True, N_SEEDS_SENS, fit6, val6,
                                     pers=te['cat_0_24'].values[mte].astype(int), verbose=False)
    add_rows('E: 24–30 h outcome', Re, y6_te, te['subject_id'].values[mte], 4); print(f'  done in {(time.time()-t0)/60:.1f} min')
sens_df = pd.DataFrame(sens_rows)
sens_tbl = sens_df[['Analysis', 'Model', 'Test stays']].copy()
for m in SENS_MET:
    sens_tbl[f'{m} (95% CI)'] = ['—' if np.isnan(r[m]) else f"{r[m]:.3f} ({r[m+' lo']:.3f}–{r[m+' hi']:.3f})" for _, r in sens_df.iterrows()]
save_table(sens_tbl, 'table_13_sensitivity_analyses'); print(sens_tbl.to_string(index=False))

MCOL = {'FT-Transformer': C_ORANGE, 'LightGBM': C_BLUE, 'XGBoost': C_AQUA, 'Random forest': '#eda100',
        'Logistic regression': '#e87ba4', 'NEWS2 persistence': C_GREY}
an_list = list(dict.fromkeys(sens_df.Analysis)); mod_list = [m for m in MCOL if m in set(sens_df.Model)]
fig, axes = plt.subplots(1, 2, figsize=(17, 5.2), sharey=False)
w = 0.8 / len(mod_list)
for ax, m, ttl in zip(axes, ['Accuracy', 'Macro-F1'], ['(a) Accuracy', '(b) Macro-F1']):
    for j, mod in enumerate(mod_list):
        sub_ = sens_df[sens_df.Model == mod].set_index('Analysis').reindex(an_list)
        x = np.arange(len(an_list)) + (j - (len(mod_list) - 1) / 2) * w
        ax.bar(x, sub_[m], width=w, color=MCOL[mod], edgecolor='white', linewidth=1, label=mod)
        ax.errorbar(x, sub_[m], yerr=[np.clip(sub_[m] - sub_[m + ' lo'], 0, None), np.clip(sub_[m + ' hi'] - sub_[m], 0, None)], fmt='none', ecolor='#0b0b0b', elinewidth=0.8, capsize=2)
    ax.set_xticks(np.arange(len(an_list))); ax.set_xticklabels([a.replace(': ', ':\n', 1) for a in an_list], fontsize=8)
    ax.set_ylim(0, 1); ax.set_ylabel(m); ax.set_title(ttl, loc='left'); ax.grid(axis='y', color='#e6e5e1'); ax.set_axisbelow(True)
h_, l_ = axes[0].get_legend_handles_labels()
fig.legend(h_, l_, loc='lower center', ncol=len(mod_list), bbox_to_anchor=(0.5, -0.06), fontsize=8)
fig.text(0.5, -0.1, 'Error bars: 95% patient-clustered bootstrap confidence intervals.', ha='center', fontsize=8, color='#52514e')
plt.tight_layout(); save_fig(fig, 'fig_8_sensitivity'); plt.show()


## 14. FT-Transformer training curve and run summary

In [ ]:
h = pd.DataFrame(hist)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for i, (sd, g_) in enumerate(h.groupby('seed')):
    kw = dict(color=C_BLUE if i == 0 else '#b7d3f4', lw=2 if i == 0 else 1, zorder=3 if i == 0 else 2)
    ax1.plot(g_.epoch, g_.train_loss, **kw)
    ax2.plot(g_.epoch, g_.val_macro_f1, **{**kw, 'color': C_ORANGE if i == 0 else '#f6c3a6'})
    ax2.scatter(seeds_df.loc[i, 'best epoch'], seeds_df.loc[i, 'val macro-F1'], color=C_ORANGE if i == 0 else '#f6c3a6', s=25, zorder=4)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Weighted cross-entropy'); ax1.set_title('(a) Training loss', loc='left')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro-F1'); ax2.set_title('(b) Validation macro-F1 (dot: selected epoch)', loc='left')
for a in (ax1, ax2): a.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
fig.suptitle(f'FT-Transformer training, {N_SEEDS} seeds (first seed highlighted)', x=0.01, ha='left', fontsize=11)
plt.tight_layout(); save_fig(fig, 'fig_s1_ftt_training'); plt.show()

fp = PARAMS['ftt']
hp = pd.DataFrame([
  ('Source', PARAM_SOURCE),
  ('FT-Transformer', f"{'PLR periodic embeddings' if FTT_EMB == 'plr' else 'linear feature tokens'} ({fp['n_freq']} frequencies, σ {fp['sigma']:.3g}) on quantile-normalised inputs; "
                     f"{fp['layers']} pre-norm blocks, d={fp['d']}, 8 heads, ReGLU FFN {fp['ffn']}, attention dropout {fp['att_drop']:.3g}, FFN dropout {fp['ffn_drop']:.3g}, "
                     f"residual dropout {fp['res_drop']:.3g}; {n_params:,} parameters per model; AdamW lr {fp['lr']:.2e}, wd {fp['wd']:.2e}, batch 256; class-weighted CE; "
                     f"early stopping on val macro-F1 (patience {FTT_PATIENCE}, max {FTT_EPOCHS}); ensemble of {N_SEEDS} seeds; "
                     f"best epochs {', '.join(str(int(e)) for e in seeds_df['best epoch'])}"),
  ('Logistic regression', 'multinomial, class_weight=balanced, max_iter 3000'),
  ('Random forest', ', '.join(f'{k}={v}' for k, v in PARAMS['rf'].items()) + ', class_weight=balanced'),
  ('XGBoost', ', '.join(f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}' for k, v in PARAMS['xgb'].items())
              + f"; balanced sample weights; early stopping (100 rounds) → {BEST_ITER.get('XGBoost')} trees"),
  ('LightGBM', ', '.join(f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}' for k, v in PARAMS['lgbm'].items())
               + f"; class_weight=balanced; early stopping (100 rounds) → {BEST_ITER.get('LightGBM')} trees"),
  ('NEWS2 persistence', 'NEWS2 category from 0–24 h medians carried forward'),
], columns=['Model', 'Settings'])
save_table(hp, 'table_s2_hyperparameters')
best_row = res.iloc[0]
print('=' * 70); print('RUN SUMMARY')
for k, v in flow.items(): print(f'  {k:45s}: {v:,}' if isinstance(v, (int, np.integer)) else f'  {k:45s}: {v}')
for _, r_ in res.iterrows():
    print(f"  {r_['Model']:22s} accuracy {r_['Accuracy']:.3f}  macro-F1 {r_['Macro-F1']:.3f} ({r_['Macro-F1 lo']:.3f}–{r_['Macro-F1 hi']:.3f})  AUROC {r_['Macro-AUROC']:.3f}")
print(f'  Spearman ρ FT-T vs LightGBM importance: {rho:.3f}')
for _, r_ in bin_df.iterrows():
    print(f"  High-or-death  {r_['Model']:22s} AUROC {r_['AUROC']:.3f} ({r_['AUROC lo']:.3f}–{r_['AUROC hi']:.3f})  AUPRC {r_['AUPRC']:.3f}")
for _, r_ in thr_df.iterrows():
    print(f"  High-or-death  {r_['Model']:22s} {r_['Rule']:32s} accuracy {r_['Accuracy']:.3f}  sens {r_['Sensitivity']:.3f}  spec {r_['Specificity']:.3f}")
for _, r_ in sens_df[sens_df.Model.isin(['FT-Transformer', BEST_BASE])].iterrows():
    print(f"  {r_['Analysis'][:34]:34s} {r_['Model']:15s} accuracy {r_['Accuracy']:.3f}  macro-F1 {r_['Macro-F1']:.3f}")
print(f'  hyperparameters: {PARAM_SOURCE}')
print('=' * 70)


In [ ]:
import zipfile
files = [f for f in os.listdir(FIG_DIR) if f.endswith(('.png', '.pdf', '.csv')) and not f.startswith('chartevents')]
with zipfile.ZipFile(os.path.join(FIG_DIR, 'results_archive.zip'), 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files: z.write(os.path.join(FIG_DIR, f), f)
print(f'results_archive.zip: {len(files)} files')
try:
    from IPython.display import FileLink, display; display(FileLink('results_archive.zip'))
except Exception: pass
